In [2]:
#Autres modules importants à utiliser
import numpy as np
import scipy.stats as stats
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from typing import Dict, List, Tuple, Any, Optional
import os
import sys
import pickle
import time
from skimage.restoration import denoise_tv_chambolle

In [1]:
!git clone https://github.com/ahxt/g-mixup.git
%cd g-mixup

Clonage dans 'g-mixup'...
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 16 (delta 3), reused 11 (delta 1), pack-reused 0 (from 0)
Réception d'objets: 100% (16/16), 1.69 Mio | 3.66 Mio/s, fait.
Résolution des deltas: 100% (3/3), fait.
/home/rbsogan/Documents/THESE/AIstat/JGS-graphon/experiments/g-mixup


/home/rbsogan/SIGL/venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [10]:
import re
import numpy as np
import pandas as pd
from io import StringIO

def parse_gmixup_logs(log_text):
    """Parse les logs de g-mixup pour extraire les métriques"""
    epochs = []
    train_losses = []
    val_losses = []
    test_losses = []
    val_accs = []
    test_accs = []

    # Pattern pour matcher les lignes d'epoch
    pattern = r'Epoch: (\d+), Train Loss: ([\d.]+), Val Loss: ([\d.]+), Test Loss: ([\d.]+),  Val Acc:  ([\d.]+), Test Acc:  ([\d.]+)'

    for line in log_text.split('\n'):
        match = re.search(pattern, line)
        if match:
            epoch = int(match.group(1))
            train_loss = float(match.group(2))
            val_loss = float(match.group(3))
            test_loss = float(match.group(4))
            val_acc = float(match.group(5))
            test_acc = float(match.group(6))

            epochs.append(epoch)
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            test_losses.append(test_loss)
            val_accs.append(val_acc)
            test_accs.append(test_acc)

    return {
        'epochs': epochs,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'test_losses': test_losses,
        'val_accs': val_accs,
        'test_accs': test_accs
    }

def find_best_epoch(metrics):
    """Trouve la meilleure époque basée sur la validation accuracy"""
    val_accs = metrics['val_accs']
    best_epoch_idx = np.argmax(val_accs)

    return {
        'best_epoch': metrics['epochs'][best_epoch_idx],
        'best_val_acc': val_accs[best_epoch_idx],
        'test_acc_at_best_epoch': metrics['test_accs'][best_epoch_idx],
        'val_loss_at_best_epoch': metrics['val_losses'][best_epoch_idx],
        'train_loss_at_best_epoch': metrics['train_losses'][best_epoch_idx]
    }

def analyze_multiple_runs(logs_list):
    """Analyse multiple runs et retourne les statistiques"""
    results = []

    for i, log_text in enumerate(logs_list):
        print(f"Analyzing run {i+1}...")
        metrics = parse_gmixup_logs(log_text)

        if len(metrics['epochs']) == 0:
            print(f"Warning: No epochs found in run {i+1}")
            continue

        best_epoch_info = find_best_epoch(metrics)
        best_epoch_info['run_id'] = i + 1
        best_epoch_info['total_epochs'] = len(metrics['epochs'])
        results.append(best_epoch_info)

    # Convertir en DataFrame pour analyse
    df = pd.DataFrame(results)

    # Statistiques finales
    stats = {
        'mean_test_acc': df['test_acc_at_best_epoch'].mean(),
        'std_test_acc': df['test_acc_at_best_epoch'].std(),
        'mean_best_epoch': df['best_epoch'].mean(),
        'mean_val_acc': df['best_val_acc'].mean(),
        'num_runs': len(df),
        'all_test_accs': df['test_acc_at_best_epoch'].tolist(),
        'all_best_epochs': df['best_epoch'].tolist()
    }

    return df, stats

def print_detailed_report(df, stats):
    """Affiche un rapport détaillé des résultats"""
    print("\n" + "="*80)
    print("RAPPORT D'ANALYSE G-MIXUP - SÉLECTION AUTOMATIQUE MEILLEURE ÉPOQUE")
    print("="*80)

    print(f"\n📊 STATISTIQUES GLOBALES ({stats['num_runs']} runs analysés):")
    print(f"   Précision test moyenne: {stats['mean_test_acc']:.4f} ± {stats['std_test_acc']:.4f}")
    print(f"   Époque moyenne de sélection: {stats['mean_best_epoch']:.1f}")
    print(f"   Précision validation moyenne: {stats['mean_val_acc']:.4f}")

    print(f"\n📈 DISTRIBUTION DES PRÉCISIONS TEST:")
    for i, acc in enumerate(stats['all_test_accs']):
        print(f"   Run {i+1}: {acc:.4f}")

    print(f"\n🎯 MEILLEURE PERFORMANCE PAR RUN:")
    for _, row in df.iterrows():
        print(f"   Run {row['run_id']}: Epoch {row['best_epoch']} - Val Acc: {row['best_val_acc']:.4f} - Test Acc: {row['test_acc_at_best_epoch']:.4f}")

# Exemple d'utilisation avec vos logs
def main():
    # Ici vous devriez charger vos logs depuis des fichiers ou les passer directement
    # Pour cet exemple, je vais créer une simulation avec vos logs

    # Simulation de multiple runs (remplacez par vos vraies données)
    logs_list = []

    # Run 1 (vos logs actuels)
    with open('/content/g-mixup_logs.txt', 'r') as f:  # Remplacez par le chemin de vos logs
        logs_list.append(f.read())

    # Ajoutez ici d'autres runs si vous en avez...
    # logs_list.append(log_text_run2)
    # logs_list.append(log_text_run3)

    # Analyse
    df, stats = analyze_multiple_runs(logs_list)

    # Rapport
    print_detailed_report(df, stats)

    # Sauvegarde des résultats
    df.to_csv('gmixup_analysis_results.csv', index=False)

    print(f"\n💾 Résultats sauvegardés dans 'gmixup_analysis_results.csv'")

    return df, stats

# Version alternative pour analyser un seul run en détail
def analyze_single_run_detailed(log_text):
    """Analyse détaillée d'un seul run"""
    metrics = parse_gmixup_logs(log_text)

    if len(metrics['epochs']) == 0:
        print("Aucune donnée d'epoch trouvée dans le log")
        return None

    best_info = find_best_epoch(metrics)

    print("\n" + "="*60)
    print("ANALYSE DÉTAILLÉE DU RUN")
    print("="*60)

    print(f"📈 Nombre total d'epochs: {len(metrics['epochs'])}")
    print(f"🎯 Meilleure epoch: {best_info['best_epoch']}")
    print(f"✅ Précision validation à la meilleure epoch: {best_info['best_val_acc']:.4f}")
    print(f"🎯 Précision test correspondante: {best_info['test_acc_at_best_epoch']:.4f}")

    # Trouver les top 5 epochs
    val_accs = np.array(metrics['val_accs'])
    top5_indices = np.argsort(val_accs)[-5:][::-1]

    print(f"\n🏆 TOP 5 EPOCHS (par validation accuracy):")
    for i, idx in enumerate(top5_indices):
        epoch = metrics['epochs'][idx]
        val_acc = val_accs[idx]
        test_acc = metrics['test_accs'][idx]
        print(f"   {i+1}. Epoch {epoch}: Val={val_acc:.4f}, Test={test_acc:.4f}")

    return best_info


In [13]:
!CUDA_VISIBLE_DEVICES=0 python -u /home/rbsogan/Documents/THESE/AIstat/JGS-graphon/experiments/jgs_gmixup.py --data_path . --model GIN --dataset IMDB-BINARY \
  --lr 0.01 --gmixup True --seed 1314 --log_screen True --batch_size 128 --num_hidden 64 \
  --aug_ratio 0.2 --aug_num 10 --ge USVT > log_file_binary_1314.txt 2>&1

In [14]:
#/content/drive/MyDrive/AIStat/g-mixup/essai_log.txt
# Exécution
if __name__ == "__main__":
    # Pour analyser vos logs actuels
    with open('log_file_binary_1314.txt', 'r') as f:  # Remplacez par votre fichier
        log_text = f.read()

    # Analyse détaillée du run actuel
    best_info = analyze_single_run_detailed(log_text)

    # Si vous avez multiple runs, décommentez:
    # df, stats = main()


ANALYSE DÉTAILLÉE DU RUN
📈 Nombre total d'epochs: 99
🎯 Meilleure epoch: 67
✅ Précision validation à la meilleure epoch: 0.7400
🎯 Précision test correspondante: 0.7550

🏆 TOP 5 EPOCHS (par validation accuracy):
   1. Epoch 67: Val=0.7400, Test=0.7550
   2. Epoch 92: Val=0.7200, Test=0.7150
   3. Epoch 66: Val=0.7100, Test=0.7650
   4. Epoch 46: Val=0.7100, Test=0.7000
   5. Epoch 72: Val=0.7000, Test=0.7650


In [16]:
!CUDA_VISIBLE_DEVICES=0 python -u /home/rbsogan/Documents/THESE/AIstat/JGS-graphon/experiments/jgs_gmixup.py --data_path . --model GIN --dataset IMDB-BINARY \
  --lr 0.01 --gmixup True --seed 11314 --log_screen True --batch_size 128 --num_hidden 64 \
  --aug_ratio 0.2 --aug_num 10 --ge JGS > log_file_binary_11314.txt 2>&1

In [17]:
#/content/drive/MyDrive/AIStat/g-mixup/essai_log.txt
# Exécution
if __name__ == "__main__":
    # Pour analyser vos logs actuels
    with open('log_file_binary_11314.txt', 'r') as f:  # Remplacez par votre fichier
        log_text = f.read()

    # Analyse détaillée du run actuel
    best_info = analyze_single_run_detailed(log_text)

    # Si vous avez multiple runs, décommentez:
    # df, stats = main()


ANALYSE DÉTAILLÉE DU RUN
📈 Nombre total d'epochs: 99
🎯 Meilleure epoch: 75
✅ Précision validation à la meilleure epoch: 0.7500
🎯 Précision test correspondante: 0.7550

🏆 TOP 5 EPOCHS (par validation accuracy):
   1. Epoch 75: Val=0.7500, Test=0.7550
   2. Epoch 17: Val=0.7400, Test=0.7250
   3. Epoch 28: Val=0.7400, Test=0.7200
   4. Epoch 13: Val=0.7400, Test=0.7250
   5. Epoch 31: Val=0.7400, Test=0.7200


In [18]:
!CUDA_VISIBLE_DEVICES=0 python -u /home/rbsogan/Documents/THESE/AIstat/JGS-graphon/experiments/jgs_gmixup.py --data_path . --model GIN --dataset IMDB-BINARY \
  --lr 0.01 --gmixup True --seed 21314 --log_screen True --batch_size 128 --num_hidden 64 \
  --aug_ratio 0.2 --aug_num 10 --ge JGS > log_file_binary_21314.txt 2>&1

In [19]:
#/content/drive/MyDrive/AIStat/g-mixup/essai_log.txt
# Exécution
if __name__ == "__main__":
    # Pour analyser vos logs actuels
    with open('log_file_binary_21314.txt', 'r') as f:  # Remplacez par votre fichier
        log_text = f.read()

    # Analyse détaillée du run actuel
    best_info = analyze_single_run_detailed(log_text)

    # Si vous avez multiple runs, décommentez:
    # df, stats = main()


ANALYSE DÉTAILLÉE DU RUN
📈 Nombre total d'epochs: 99
🎯 Meilleure epoch: 36
✅ Précision validation à la meilleure epoch: 0.8000
🎯 Précision test correspondante: 0.6750

🏆 TOP 5 EPOCHS (par validation accuracy):
   1. Epoch 41: Val=0.8000, Test=0.7050
   2. Epoch 36: Val=0.8000, Test=0.6750
   3. Epoch 72: Val=0.7900, Test=0.6550
   4. Epoch 95: Val=0.7900, Test=0.6450
   5. Epoch 42: Val=0.7900, Test=0.7350


In [20]:
!CUDA_VISIBLE_DEVICES=0 python -u /home/rbsogan/Documents/THESE/AIstat/JGS-graphon/experiments/jgs_gmixup.py --data_path . --model GIN --dataset IMDB-BINARY \
  --lr 0.01 --gmixup True --seed 31314 --log_screen True --batch_size 128 --num_hidden 64 \
  --aug_ratio 0.2 --aug_num 10 --ge JGS > log_file_binary_31314.txt 2>&1

^C


In [21]:
#/content/drive/MyDrive/AIStat/g-mixup/essai_log.txt
# Exécution
if __name__ == "__main__":
    # Pour analyser vos logs actuels
    with open('log_file_binary_31314.txt', 'r') as f:  # Remplacez par votre fichier
        log_text = f.read()

    # Analyse détaillée du run actuel
    best_info = analyze_single_run_detailed(log_text)

    # Si vous avez multiple runs, décommentez:
    # df, stats = main()


ANALYSE DÉTAILLÉE DU RUN
📈 Nombre total d'epochs: 246
🎯 Meilleure epoch: 183
✅ Précision validation à la meilleure epoch: 0.8100
🎯 Précision test correspondante: 0.7300

🏆 TOP 5 EPOCHS (par validation accuracy):
   1. Epoch 183: Val=0.8100, Test=0.7300
   2. Epoch 65: Val=0.7900, Test=0.7350
   3. Epoch 64: Val=0.7900, Test=0.7300
   4. Epoch 15: Val=0.7800, Test=0.7050
   5. Epoch 131: Val=0.7800, Test=0.7600


In [26]:
!CUDA_VISIBLE_DEVICES=0 python -u /home/rbsogan/Documents/THESE/AIstat/JGS-graphon/experiments/jgs_gmixup.py --data_path . --model GIN --dataset IMDB-BINARY \
  --lr 0.01 --gmixup True --seed 41314 --log_screen True --batch_size 128 --num_hidden 64 \
  --aug_ratio 0.2 --aug_num 10 --ge JGS > log_file_binary_41314.txt 2>&1

In [27]:
#/content/drive/MyDrive/AIStat/g-mixup/essai_log.txt
# Exécution
if __name__ == "__main__":
    # Pour analyser vos logs actuels
    with open('log_file_binary_41314.txt', 'r') as f:  # Remplacez par votre fichier
        log_text = f.read()

    # Analyse détaillée du run actuel
    best_info = analyze_single_run_detailed(log_text)

    # Si vous avez multiple runs, décommentez:
    # df, stats = main()


ANALYSE DÉTAILLÉE DU RUN
📈 Nombre total d'epochs: 99
🎯 Meilleure epoch: 57
✅ Précision validation à la meilleure epoch: 0.7800
🎯 Précision test correspondante: 0.7350

🏆 TOP 5 EPOCHS (par validation accuracy):
   1. Epoch 57: Val=0.7800, Test=0.7350
   2. Epoch 77: Val=0.7700, Test=0.7550
   3. Epoch 37: Val=0.7700, Test=0.7750
   4. Epoch 94: Val=0.7600, Test=0.7600
   5. Epoch 6: Val=0.7500, Test=0.7500


In [28]:
!CUDA_VISIBLE_DEVICES=0 python -u /home/rbsogan/Documents/THESE/AIstat/JGS-graphon/experiments/jgs_gmixup.py --data_path . --model GIN --dataset IMDB-BINARY \
  --lr 0.01 --gmixup True --seed 51314 --log_screen True --batch_size 128 --num_hidden 64 \
  --aug_ratio 0.2 --aug_num 10 --ge JGS > log_file_binary_51314.txt 2>&1

In [29]:
#/content/drive/MyDrive/AIStat/g-mixup/essai_log.txt
# Exécution
if __name__ == "__main__":
    # Pour analyser vos logs actuels
    with open('log_file_binary_51314.txt', 'r') as f:  # Remplacez par votre fichier
        log_text = f.read()

    # Analyse détaillée du run actuel
    best_info = analyze_single_run_detailed(log_text)

    # Si vous avez multiple runs, décommentez:
    # df, stats = main()


ANALYSE DÉTAILLÉE DU RUN
📈 Nombre total d'epochs: 99
🎯 Meilleure epoch: 70
✅ Précision validation à la meilleure epoch: 0.8300
🎯 Précision test correspondante: 0.7100

🏆 TOP 5 EPOCHS (par validation accuracy):
   1. Epoch 70: Val=0.8300, Test=0.7100
   2. Epoch 89: Val=0.8300, Test=0.6950
   3. Epoch 76: Val=0.8300, Test=0.7150
   4. Epoch 27: Val=0.8200, Test=0.7350
   5. Epoch 39: Val=0.8100, Test=0.6950


In [30]:
!CUDA_VISIBLE_DEVICES=0 python -u /home/rbsogan/Documents/THESE/AIstat/JGS-graphon/experiments/jgs_gmixup.py --data_path . --model GIN --dataset IMDB-BINARY \
  --lr 0.01 --gmixup True --seed 61314 --log_screen True --batch_size 128 --num_hidden 64 \
  --aug_ratio 0.2 --aug_num 10 --ge JGS > log_file_binary_61314.txt 2>&1

In [31]:
#/content/drive/MyDrive/AIStat/g-mixup/essai_log.txt
# Exécution
if __name__ == "__main__":
    # Pour analyser vos logs actuels
    with open('log_file_binary_61314.txt', 'r') as f:  # Remplacez par votre fichier
        log_text = f.read()

    # Analyse détaillée du run actuel
    best_info = analyze_single_run_detailed(log_text)

    # Si vous avez multiple runs, décommentez:
    # df, stats = main()


ANALYSE DÉTAILLÉE DU RUN
📈 Nombre total d'epochs: 99
🎯 Meilleure epoch: 19
✅ Précision validation à la meilleure epoch: 0.7800
🎯 Précision test correspondante: 0.7200

🏆 TOP 5 EPOCHS (par validation accuracy):
   1. Epoch 19: Val=0.7800, Test=0.7200
   2. Epoch 50: Val=0.7600, Test=0.6850
   3. Epoch 20: Val=0.7400, Test=0.7400
   4. Epoch 64: Val=0.7300, Test=0.6800
   5. Epoch 8: Val=0.7300, Test=0.7800


In [34]:
!CUDA_VISIBLE_DEVICES=0 python -u /home/rbsogan/Documents/THESE/AIstat/JGS-graphon/experiments/jgs_gmixup.py --data_path . --model GIN --dataset IMDB-BINARY \
  --lr 0.01 --gmixup True --seed 71314 --log_screen True --batch_size 128 --num_hidden 64 \
  --aug_ratio 0.2 --aug_num 10 --ge JGS > log_file_binary_71314.txt 2>&1

In [33]:
#/content/drive/MyDrive/AIStat/g-mixup/essai_log.txt
# Exécution
if __name__ == "__main__":
    # Pour analyser vos logs actuels
    with open('log_file_binary_71314.txt', 'r') as f:  # Remplacez par votre fichier
        log_text = f.read()

    # Analyse détaillée du run actuel
    best_info = analyze_single_run_detailed(log_text)

    # Si vous avez multiple runs, décommentez:
    # df, stats = main()


ANALYSE DÉTAILLÉE DU RUN
📈 Nombre total d'epochs: 99
🎯 Meilleure epoch: 86
✅ Précision validation à la meilleure epoch: 0.7500
🎯 Précision test correspondante: 0.7100

🏆 TOP 5 EPOCHS (par validation accuracy):
   1. Epoch 86: Val=0.7500, Test=0.7100
   2. Epoch 16: Val=0.7400, Test=0.7150
   3. Epoch 31: Val=0.7400, Test=0.6500
   4. Epoch 43: Val=0.7400, Test=0.7600
   5. Epoch 44: Val=0.7300, Test=0.7550
